# **64D 128D, 256D comparisons**

In [1]:
# =============================================================================
# URDU NEWS HEADLINE EMBEDDINGS WITH MULTI-DIMENSIONAL PCA
# Creates ChromaDB collections: Full (768D), PCA-64D, PCA-128D, PCA-256D
# Uses CLS POOLING on HEADLINE column
# Compares top 15 results and query latency across all dimensions
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from pathlib import Path
import time
from sklearn.decomposition import PCA
import pickle
import matplotlib.pyplot as plt
from matplotlib import rcParams
import warnings
warnings.filterwarnings('ignore')

# Configure matplotlib
rcParams['figure.figsize'] = (16, 10)
rcParams['font.size'] = 10

class MultiDimensionalHeadlineEmbedder:
    """
    Generate headline embeddings with multiple PCA dimensions using CLS pooling.
    Creates 4 collections: Full (768D), PCA-64D, PCA-128D, PCA-256D
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 base_path: str = "./chroma_db_headline_collections",
                 pca_dimensions: list = [64, 128, 256]):
        """
        Initialize with multiple PCA dimensions.

        Args:
            model_name: HuggingFace model identifier
            base_path: Base path for all ChromaDB collections
            pca_dimensions: List of PCA dimensions to create
        """
        self.model_name = model_name
        self.base_path = Path(base_path)
        self.pca_dimensions = pca_dimensions
        self.pca_models = {}

        # Create base directory
        self.base_path.mkdir(exist_ok=True)

        # Setup device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

        # Initialize ChromaDB clients and collections
        self.clients = {}
        self.collections = {}

        # Full embeddings collection
        full_path = self.base_path / "chroma_db_headline_full_768D"
        full_path.mkdir(exist_ok=True)
        print(f"Initializing Full Headline Embeddings (768D) at: {full_path}")
        self.clients['full'] = chromadb.PersistentClient(path=str(full_path))
        self.collections['full'] = self.clients['full'].get_or_create_collection(
            name="urdu_news_headline_full_768D_cls",
            metadata={"hnsw:space": "cosine"}
        )

        # PCA collections
        for dim in pca_dimensions:
            pca_path = self.base_path / f"chroma_db_headline_pca_{dim}D"
            pca_path.mkdir(exist_ok=True)
            print(f"Initializing PCA-{dim}D Headline at: {pca_path}")
            self.clients[f'pca_{dim}'] = chromadb.PersistentClient(path=str(pca_path))
            self.collections[f'pca_{dim}'] = self.clients[f'pca_{dim}'].get_or_create_collection(
                name=f"urdu_news_headline_pca_{dim}D_cls",
                metadata={"hnsw:space": "cosine"}
            )

    def cls_pooling(self, model_output, attention_mask=None):
        """
        Apply CLS pooling - extract [CLS] token embedding.

        Args:
            model_output: Output from transformer model
            attention_mask: Not used in CLS pooling

        Returns:
            CLS token embeddings (batch_size, embedding_dim)
        """
        token_embeddings = model_output[0]
        # Extract [CLS] token (first token)
        cls_embeddings = token_embeddings[:, 0, :]
        return cls_embeddings

    def generate_embedding_for_text(self, text: str, max_length: int = 512) -> np.ndarray:
        """Generate embedding for headline using CLS pooling."""
        encoded_input = self.tokenizer(
            text,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt',
            return_attention_mask=True,
            add_special_tokens=True
        )
        encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

        with torch.no_grad():
            model_output = self.model(**encoded_input)

        embeddings = self.cls_pooling(model_output, encoded_input['attention_mask'])
        return embeddings.cpu().detach().numpy()[0]

    def fit_pca_models(self, embeddings_array: np.ndarray) -> None:
        """Fit PCA models for all dimensions."""
        print(f"\n{'='*70}")
        print(f"FITTING PCA MODELS FOR DIMENSIONS: {self.pca_dimensions}")
        print(f"{'='*70}")
        print(f"Input shape: {embeddings_array.shape}")

        for dim in self.pca_dimensions:
            print(f"\nFitting PCA-{dim}D...")
            pca = PCA(n_components=dim, random_state=42)
            pca.fit(embeddings_array)
            self.pca_models[dim] = pca

            explained_var = np.sum(pca.explained_variance_ratio_) * 100
            print(f"  ✓ Explained variance: {explained_var:.2f}%")

            # Save PCA model
            pca_path = self.base_path / f"chroma_db_headline_pca_{dim}D" / "pca_model.pkl"
            with open(pca_path, 'wb') as f:
                pickle.dump(pca, f)
            print(f"  ✓ Model saved to: {pca_path}")

    def apply_pca(self, embeddings_array: np.ndarray, dimension: int) -> np.ndarray:
        """Apply PCA transformation for specific dimension."""
        if dimension not in self.pca_models:
            raise ValueError(f"PCA model for {dimension}D not fitted.")
        return self.pca_models[dimension].transform(embeddings_array)

    def generate_embeddings_for_dataset(self, df: pd.DataFrame,
                                       headline_column: str = "Headline",
                                       content_column: str = "content",
                                       category_column: str = "Category") -> None:
        """Generate and store headline embeddings in all collections."""
        print(f"\n{'='*70}")
        print(f"GENERATING HEADLINE EMBEDDINGS FOR {len(df)} ARTICLES")
        print(f"{'='*70}")
        print(f"Using CLS POOLING on '{headline_column}' column")

        total_articles = len(df)
        start_time = time.time()

        ids = []
        embeddings_full = []
        metadatas = []
        documents = []

        # Step 1: Generate full embeddings from headlines
        print("\nSTEP 1: Generating full embeddings from headlines (768D)...")
        print("="*70)

        for idx, row in df.iterrows():
            if (idx + 1) % 500 == 0:
                elapsed = time.time() - start_time
                print(f"Processed {idx + 1}/{total_articles} articles ({elapsed:.2f}s)")

            headline_text = str(row[headline_column])
            if len(headline_text.strip()) == 0:
                continue

            try:
                embedding = self.generate_embedding_for_text(headline_text)

                ids.append(f"article_{idx}")
                embeddings_full.append(embedding)
                documents.append(headline_text)

                metadatas.append({
                    "article_index": idx,
                    "headline": headline_text,
                    "category": str(row.get(category_column, "Unknown")),
                    "content": str(row.get(content_column, ""))[:500],
                    "content_length": len(str(row.get(content_column, ""))),
                    "pooling_method": "cls_pooling",
                    "embedding_source": "headline"
                })
            except Exception as e:
                print(f"Error processing article {idx}: {str(e)}")
                continue

        embeddings_full_array = np.array(embeddings_full)
        print(f"✓ Generated {len(embeddings_full)} headline embeddings")

        # Step 2: Fit PCA models
        self.fit_pca_models(embeddings_full_array)

        # Step 3: Store full embeddings
        print(f"\n{'='*70}")
        print("STEP 3: Storing full headline embeddings (768D)...")
        print("="*70)
        self._store_embeddings('full', ids, embeddings_full_array, documents, metadatas, 768)

        # Step 4: Store PCA embeddings for all dimensions
        for dim in self.pca_dimensions:
            print(f"\n{'='*70}")
            print(f"STEP 4.{self.pca_dimensions.index(dim)+1}: Applying and storing PCA-{dim}D...")
            print("="*70)

            embeddings_pca = self.apply_pca(embeddings_full_array, dim)
            self._store_embeddings(f'pca_{dim}', ids, embeddings_pca, documents, metadatas, dim)

        total_time = time.time() - start_time
        print(f"\n{'='*70}")
        print("✓ ALL HEADLINE EMBEDDINGS GENERATED AND STORED!")
        print(f"{'='*70}")
        print(f"Total embeddings: {len(ids)}")
        print(f"Embedding source: HEADLINE column (CLS pooling)")
        print(f"Collections created:")
        print(f"  - Full (768D): {len(ids)} embeddings")
        for dim in self.pca_dimensions:
            print(f"  - PCA-{dim}D: {len(ids)} embeddings")
        print(f"Total time: {total_time:.2f}s ({total_time/60:.2f} min)")

    def _store_embeddings(self, collection_key: str, ids: list, embeddings: np.ndarray,
                         documents: list, metadatas: list, dimension: int) -> None:
        """Helper to store embeddings in batches."""
        batch_size = 5000
        collection = self.collections[collection_key]

        for batch_idx in range(0, len(ids), batch_size):
            batch_end = min(batch_idx + batch_size, len(ids))
            print(f"  Storing items {batch_idx} to {batch_end}...")

            batch_metadatas = [
                {**meta, "embedding_type": collection_key, "dimensions": dimension}
                for meta in metadatas[batch_idx:batch_end]
            ]

            collection.add(
                ids=ids[batch_idx:batch_end],
                embeddings=[emb.tolist() for emb in embeddings[batch_idx:batch_end]],
                documents=documents[batch_idx:batch_end],
                metadatas=batch_metadatas
            )
        print(f"  ✓ Stored successfully!")

    def search_with_latency(self, query_text: str, collection_key: str,
                           n_results: int = 15) -> tuple:
        """Search and measure query latency."""
        # Generate query embedding
        query_embedding = self.generate_embedding_for_text(query_text)

        # Apply PCA if needed
        if collection_key.startswith('pca_'):
            dim = int(collection_key.split('_')[1])
            if dim not in self.pca_models:
                # Load PCA model
                pca_path = self.base_path / f"chroma_db_headline_pca_{dim}D" / "pca_model.pkl"
                with open(pca_path, 'rb') as f:
                    self.pca_models[dim] = pickle.load(f)
            query_embedding = self.pca_models[dim].transform(query_embedding.reshape(1, -1))[0]

        # Measure query latency
        start_time = time.time()
        results = self.collections[collection_key].query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results
        )
        latency = (time.time() - start_time) * 1000  # Convert to milliseconds

        return results, latency

    def compare_results_comprehensive(self, queries: list, output_dir: str = "./headline_comparison_results"):
        """Comprehensive comparison of all headline embeddings with top 15 results."""
        output_path = Path(output_dir)
        output_path.mkdir(exist_ok=True)

        print(f"\n{'='*70}")
        print("COMPREHENSIVE HEADLINE COMPARISON: TOP 15 RESULTS ACROSS ALL DIMENSIONS")
        print(f"{'='*70}")

        all_comparisons = []

        for query_idx, query in enumerate(queries, 1):
            print(f"\n{'='*70}")
            print(f"QUERY {query_idx}: {query[:80]}...")
            print(f"{'='*70}")

            # Search in all collections
            results_full, latency_full = self.search_with_latency(query, 'full', n_results=15)
            ids_full = set(results_full['ids'][0])

            print(f"\n✓ Full (768D) - Latency: {latency_full:.2f}ms")
            print(f"  Top 5 headlines:")
            for i in range(min(5, len(results_full['ids'][0]))):
                headline = results_full['metadatas'][0][i].get('headline', 'N/A')
                score = 1 - results_full['distances'][0][i]
                print(f"    {i+1}. [{score:.4f}] {headline[:60]}...")

            comparison_data = {
                'query': query,
                'query_idx': query_idx,
                'full_latency': latency_full,
                'full_ids': ids_full,
                'pca_results': {}
            }

            # Compare with each PCA dimension
            for dim in self.pca_dimensions:
                results_pca, latency_pca = self.search_with_latency(
                    query, f'pca_{dim}', n_results=15
                )
                ids_pca = set(results_pca['ids'][0])

                overlap = len(ids_full.intersection(ids_pca))
                overlap_pct = (overlap / 15) * 100

                print(f"\n✓ PCA-{dim}D - Latency: {latency_pca:.2f}ms")
                print(f"  Overlap with Full: {overlap}/15 ({overlap_pct:.1f}%)")
                print(f"  Top 5 headlines:")
                for i in range(min(5, len(results_pca['ids'][0]))):
                    headline = results_pca['metadatas'][0][i].get('headline', 'N/A')
                    score = 1 - results_pca['distances'][0][i]
                    match = "✓" if results_pca['ids'][0][i] in ids_full else "✗"
                    print(f"    {i+1}. [{score:.4f}] {match} {headline[:60]}...")

                comparison_data['pca_results'][dim] = {
                    'latency': latency_pca,
                    'ids': ids_pca,
                    'overlap': overlap,
                    'overlap_pct': overlap_pct,
                    'results': results_pca
                }

            all_comparisons.append(comparison_data)

            # Create comparison visualizations for this query
            self._create_query_visualizations(comparison_data, results_full, output_path)

        # Create summary visualizations
        self._create_summary_visualizations(all_comparisons, output_path)

        return all_comparisons

    def _create_query_visualizations(self, comparison_data: dict, results_full: dict,
                                     output_path: Path):
        """Create visualizations for one query."""
        query_idx = comparison_data['query_idx']

        # Overlap pie charts
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f"Query {query_idx}: Headline Overlap with Full (768D) - Top 15 Results\n" +
                     f"Query: {comparison_data['query'][:80]}...",
                     fontsize=14, fontweight='bold')

        for idx, dim in enumerate(self.pca_dimensions):
            ax = axes[idx]
            pca_data = comparison_data['pca_results'][dim]

            overlap = pca_data['overlap']
            non_overlap = 15 - overlap

            sizes = [overlap, non_overlap]
            colors = ['#2ecc71', '#e74c3c']
            labels = [f'Overlap: {overlap}/15\n({pca_data["overlap_pct"]:.1f}%)',
                     f'Different: {non_overlap}/15']
            explode = (0.05, 0)

            ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
                  startangle=90, explode=explode, textprops={'fontsize': 10})
            ax.set_title(f'Full vs PCA-{dim}D\nLatency: {pca_data["latency"]:.2f}ms',
                        fontsize=12, fontweight='bold')

        plt.tight_layout()
        plt.savefig(output_path / f'headline_query_{query_idx}_comparison.png',
                   dpi=300, bbox_inches='tight')
        plt.close()

        print(f"  ✓ Saved: headline_query_{query_idx}_comparison.png")

        # Create detailed text comparison
        self._create_text_comparison(comparison_data, results_full, output_path)

    def _create_text_comparison(self, comparison_data: dict, results_full: dict,
                               output_path: Path):
        """Create detailed text comparison file."""
        query_idx = comparison_data['query_idx']
        output_file = output_path / f'headline_query_{query_idx}_details.txt'

        with open(output_file, 'w', encoding='utf-8') as f:
            f.write("="*100 + "\n")
            f.write(f"HEADLINE RECOMMENDATION COMPARISON - QUERY {query_idx}\n")
            f.write("="*100 + "\n\n")
            f.write(f"Query: {comparison_data['query']}\n")
            f.write(f"Top 15 Headlines Comparison\n")
            f.write("="*100 + "\n\n")

            for i in range(15):
                f.write(f"RANK {i+1}\n")
                f.write("-"*100 + "\n")

                # Full embedding result
                if i < len(results_full['ids'][0]):
                    headline = results_full['metadatas'][0][i].get('headline', 'N/A')
                    score = 1 - results_full['distances'][0][i]
                    f.write(f"Full (768D): [{score:.4f}] {headline}\n")
                else:
                    f.write(f"Full (768D): N/A\n")

                # PCA results
                for dim in self.pca_dimensions:
                    pca_results = comparison_data['pca_results'][dim]['results']
                    if i < len(pca_results['ids'][0]):
                        headline = pca_results['metadatas'][0][i].get('headline', 'N/A')
                        score = 1 - pca_results['distances'][0][i]
                        match = "✓" if pca_results['ids'][0][i] == results_full['ids'][0][i] else "✗"
                        f.write(f"PCA-{dim}D:  [{score:.4f}] {match} {headline}\n")
                    else:
                        f.write(f"PCA-{dim}D: N/A\n")

                f.write("\n")

        print(f"  ✓ Saved: headline_query_{query_idx}_details.txt")

    def _create_summary_visualizations(self, all_comparisons: list, output_path: Path):
        """Create summary visualizations across all queries."""
        print(f"\n{'='*70}")
        print("CREATING SUMMARY VISUALIZATIONS")
        print(f"{'='*70}")

        # Calculate averages
        avg_latencies = {'full': np.mean([c['full_latency'] for c in all_comparisons])}
        avg_overlaps = {}

        for dim in self.pca_dimensions:
            latencies = [c['pca_results'][dim]['latency'] for c in all_comparisons]
            overlaps = [c['pca_results'][dim]['overlap_pct'] for c in all_comparisons]
            avg_latencies[f'pca_{dim}'] = np.mean(latencies)
            avg_overlaps[dim] = np.mean(overlaps)

        # Create summary figure
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Plot 1: Average Query Latency
        ax1 = axes[0]
        dims = ['Full\n(768D)'] + [f'PCA-{d}D' for d in self.pca_dimensions]
        latencies = [avg_latencies['full']] + [avg_latencies[f'pca_{d}'] for d in self.pca_dimensions]
        colors = ['#3498db', '#e67e22', '#9b59b6', '#1abc9c']

        bars = ax1.bar(dims, latencies, color=colors, edgecolor='black', linewidth=1.5)
        ax1.set_ylabel('Average Latency (ms)', fontsize=12, fontweight='bold')
        ax1.set_title('Average Query Latency - Headline Embeddings',
                     fontsize=13, fontweight='bold')
        ax1.grid(axis='y', alpha=0.3, linestyle='--')

        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}ms', ha='center', va='bottom',
                    fontsize=10, fontweight='bold')

        # Plot 2: Average Overlap Percentage
        ax2 = axes[1]
        dims_pca = [f'PCA-{d}D' for d in self.pca_dimensions]
        overlaps = [avg_overlaps[d] for d in self.pca_dimensions]
        colors_pca = ['#e67e22', '#9b59b6', '#1abc9c']

        bars = ax2.bar(dims_pca, overlaps, color=colors_pca, edgecolor='black', linewidth=1.5)
        ax2.set_ylabel('Average Overlap (%)', fontsize=12, fontweight='bold')
        ax2.set_title('Average Headline Overlap with Full (Top 15)',
                     fontsize=13, fontweight='bold')
        ax2.set_ylim([0, 100])
        ax2.grid(axis='y', alpha=0.3, linestyle='--')
        ax2.axhline(y=100, color='green', linestyle='--', linewidth=2, alpha=0.5)

        for bar in bars:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}%', ha='center', va='bottom',
                    fontsize=10, fontweight='bold')

        fig.suptitle(f'Summary Statistics: Headline Embeddings ({len(all_comparisons)} Queries)',
                    fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(output_path / 'headline_summary_comparison.png', dpi=300, bbox_inches='tight')
        plt.close()

        print("✓ Saved: headline_summary_comparison.png")

        # Print summary statistics
        print(f"\n{'='*70}")
        print("HEADLINE EMBEDDINGS SUMMARY STATISTICS")
        print(f"{'='*70}")
        print(f"Total queries analyzed: {len(all_comparisons)}\n")
        print("Average Query Latencies:")
        print(f"  Full (768D): {avg_latencies['full']:.2f}ms")
        for dim in self.pca_dimensions:
            speedup = avg_latencies['full'] / avg_latencies[f'pca_{dim}']
            print(f"  PCA-{dim}D: {avg_latencies[f'pca_{dim}']:.2f}ms ({speedup:.2f}x)")

        print(f"\nAverage Overlap with Full (Top 15):")
        for dim in self.pca_dimensions:
            print(f"  PCA-{dim}D: {avg_overlaps[dim]:.1f}%")


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("MULTI-DIMENSIONAL PCA URDU NEWS HEADLINE EMBEDDINGS")
    print("Dimensions: Full (768D), PCA-64D, PCA-128D, PCA-256D")
    print("Using CLS POOLING on HEADLINE column")
    print("="*70)

    # Load dataset
    print("\nLoading dataset...")
    df = pd.read_csv("final_cleaned_urdu_news.csv")
    print(f"✓ Dataset loaded: {df.shape[0]} articles")
    print(f"  Categories: {df['Category'].unique().tolist()}")
    print(f"  Sample headline: {df.iloc[0]['Headline']}")

    # Initialize embedder
    print(f"\n{'='*70}")
    print("INITIALIZING MULTI-DIMENSIONAL HEADLINE EMBEDDER")
    print(f"{'='*70}")

    embedder = MultiDimensionalHeadlineEmbedder(
        model_name="urduhack/roberta-urdu-small",
        base_path="./chroma_db_headline_collections",
        pca_dimensions=[64, 128, 256]
    )

    # Generate embeddings (comment out if already generated)
    embedder.generate_embeddings_for_dataset(
        df=df,
        headline_column="Headline",
        content_column="content",
        category_column="Category"
    )

    # Define test queries (headlines)
    test_queries = [
        "پاکستان میں موبائل کمپنیاں مقامی طور پر اسمبلنگ",
        "سیاسی جماعتوں کے درمیان مذاکرات",
        "کرکٹ ٹیم کی شاندار کارکردگی",
        "معیشت میں بہتری کے آثار",
        "تعلیمی اداروں میں نئی پالیسیوں کا اعلان"
    ]

    # Run comprehensive comparison
    print(f"\n{'='*70}")
    print("STARTING COMPREHENSIVE HEADLINE COMPARISON")
    print(f"{'='*70}")

    comparisons = embedder.compare_results_comprehensive(
        queries=test_queries,
        output_dir="./headline_comparison_results"
    )

    print(f"\n{'='*70}")
    print("✓ ALL HEADLINE COMPARISONS COMPLETED!")
    print(f"{'='*70}")
    print(f"Results saved in: ./headline_comparison_results/")
    print(f"  - Individual query comparisons: headline_query_1_comparison.png, etc.")
    print(f"  - Detailed text files: headline_query_1_details.txt, etc.")
    print(f"  - Summary statistics: headline_summary_comparison.png")
    print(f"\nEmbedding Method: CLS POOLING on HEADLINE column")
    print(f"Comparison: Full (768D) vs PCA-64D, PCA-128D, PCA-256D")

MULTI-DIMENSIONAL PCA URDU NEWS HEADLINE EMBEDDINGS
Dimensions: Full (768D), PCA-64D, PCA-128D, PCA-256D
Using CLS POOLING on HEADLINE column

Loading dataset...
✓ Dataset loaded: 111853 articles
  Categories: ['Business & Economics', 'Entertainment', 'Science & Technology', 'Sports', nan]
  Sample headline: عالمی بینک عسکریت پسندی سے متاثرہ خاندانوں کی معاونت کرے گا

INITIALIZING MULTI-DIMENSIONAL HEADLINE EMBEDDER
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Initializing Full Headline Embeddings (768D) at: chroma_db_headline_collections/chroma_db_headline_full_768D
Initializing PCA-64D Headline at: chroma_db_headline_collections/chroma_db_headline_pca_64D
Initializing PCA-128D Headline at: chroma_db_headline_collections/chroma_db_headline_pca_128D
Initializing PCA-256D Headline at: chroma_db_headline_collections/chroma_db_headline_pca_256D

GENERATING HEADLINE EMBEDDINGS FOR 111853 ARTICLES
Using CLS POOLING on 'Headline' column

STEP 1: Generating full embeddings fr

In [2]:
# =============================================================================
# MULTI-DIMENSIONAL URDU NEWS RECOMMENDATION SYSTEM WITH CONTENT EXTRACTION
# Compares recommendations from Full (768D) vs PCA (64D, 128D, 256D) embeddings
# Extracts article content (minimum 250 characters) for each recommendation
# Saves all results to a comprehensive DOCX file using python-docx
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import seaborn as sns# =============================================================================
# MULTI-DIMENSIONAL URDU NEWS RECOMMENDATION SYSTEM WITH CONTENT EXTRACTION
# Compares recommendations from Full (768D) vs PCA (64D, 128D, 256D) embeddings
# Extracts article content (minimum 250 characters) for each recommendation
# Saves ONLY 64D results to DOCX file using python-docx
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import time
import warnings
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
warnings.filterwarnings('ignore')

# Configure matplotlib
rcParams['figure.figsize'] = (16, 10)
rcParams['font.size'] = 10
sns.set_style("whitegrid")

class MultiDimensionalUrduNewsRecommender:
    """
    Recommendation system comparing Full (768D) with PCA-reduced embeddings (64D, 128D, 256D).
    Uses CLS pooling for embeddings.
    Includes query latency measurements and comprehensive overlap analysis.
    Extracts article content for each recommendation.
    Saves ONLY 64D results to DOCX.
    """

    def __init__(self,
                 model_name: str = "urduhack/roberta-urdu-small",
                 base_path: str = "./chroma_db_headline_collections",
                 pca_dimensions: list = [64, 128, 256]):
        """
        Initialize recommender with connections to all embedding databases.

        Args:
            model_name: HuggingFace model identifier
            base_path: Base path for all ChromaDB collections
            pca_dimensions: List of PCA dimensions to compare
        """
        self.model_name = model_name
        self.base_path = Path(base_path)
        self.pca_dimensions = pca_dimensions
        self.pca_models = {}

        # Setup device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

        # Load PCA models
        for dim in pca_dimensions:
            pca_path = self.base_path / f"chroma_db_headline_pca_{dim}D" / "pca_model.pkl"
            if pca_path.exists():
                print(f"Loading PCA-{dim}D model from: {pca_path}")
                with open(pca_path, 'rb') as f:
                    self.pca_models[dim] = pickle.load(f)
                print(f"  ✓ PCA-{dim}D model loaded")
            else:
                print(f"  ⚠ Warning: PCA-{dim}D model not found")

        # Initialize ChromaDB clients and collections
        self.clients = {}
        self.collections = {}

        # Full embeddings collection
        full_path = self.base_path / "chroma_db_headline_full_768D"
        if full_path.exists():
            print(f"\nConnecting to Full Headline Embeddings (768D) at: {full_path}")
            self.clients['full'] = chromadb.PersistentClient(path=str(full_path))
            try:
                self.collections['full'] = self.clients['full'].get_collection(name="urdu_news_headline_full_768D_cls")
                print(f"  ✓ Connected: {self.collections['full'].count()} articles")
            except Exception as e:
                print(f"  ✗ Error: {e}")

        # PCA collections
        for dim in pca_dimensions:
            pca_path = self.base_path / f"chroma_db_headline_pca_{dim}D"
            if pca_path.exists():
                print(f"Connecting to PCA-{dim}D Headline at: {pca_path}")
                self.clients[f'pca_{dim}'] = chromadb.PersistentClient(path=str(pca_path))
                try:
                    self.collections[f'pca_{dim}'] = self.clients[f'pca_{dim}'].get_collection(
                        name=f"urdu_news_headline_pca_{dim}D_cls"
                    )
                    print(f"  ✓ Connected: {self.collections[f'pca_{dim}'].count()} articles")
                except Exception as e:
                    print(f"  ✗ Error: {e}")

    def cls_pooling(self, model_output, attention_mask=None):
        """
        Apply CLS pooling to get sentence embeddings.
        Returns the embedding of the [CLS] token.
        """
        return model_output.last_hidden_state[:, 0, :]

    def generate_query_embedding(self, query_text: str, max_length: int = 512,
                                 chunk_overlap: int = 50, apply_pca: bool = False,
                                 pca_dim: int = None) -> np.ndarray:
        """
        Generate embedding for query text using CLS pooling.
        Optionally applies PCA dimensionality reduction.

        Args:
            query_text: Urdu query text
            max_length: Maximum tokens per chunk
            chunk_overlap: Overlap between chunks
            apply_pca: If True, apply PCA reduction
            pca_dim: Which PCA dimension to apply (64, 128, or 256)

        Returns:
            Query embedding vector
        """
        tokens = self.tokenizer.encode(query_text, add_special_tokens=True)

        # Process short queries
        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                query_text, padding=True, truncation=True,
                max_length=max_length, return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.cls_pooling(model_output)
            embedding = embeddings.cpu().detach().numpy()[0]

            # Apply PCA if requested
            if apply_pca and pca_dim in self.pca_models:
                embedding = self.pca_models[pca_dim].transform(embedding.reshape(1, -1))[0]

            return embedding

        # For long queries: chunking
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text, padding=True, truncation=True,
                max_length=max_length, return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.cls_pooling(model_output)
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        final_embedding = np.mean(chunk_embeddings, axis=0)

        # Apply PCA if requested
        if apply_pca and pca_dim in self.pca_models:
            final_embedding = self.pca_models[pca_dim].transform(final_embedding.reshape(1, -1))[0]

        return final_embedding

    def get_article_content(self, article_id: str, metadata: dict, min_chars: int = 250) -> str:
        """
        Extract article content with minimum character requirement.

        Args:
            article_id: Article identifier
            metadata: Article metadata dictionary
            min_chars: Minimum characters to extract (default 250)

        Returns:
            Article content (at least min_chars if available)
        """
        # Try to get content from metadata
        content = metadata.get('content', '')
        
        # If content is empty, try to construct from available fields
        if not content:
            headline = metadata.get('headline', '')
            description = metadata.get('description', '')
            text = metadata.get('text', '')
            
            # Combine available text fields
            content = f"{headline}\n\n{description}\n\n{text}".strip()
        
        # If total content is less than min_chars, return what we have
        if len(content) < min_chars:
            return content if content else "محتویٰ دستیاب نہیں (Content not available)"
        
        # Return at least min_chars characters (or full content if it's close)
        return content

    def get_recommendations_with_content(self, query: str, collection_key: str,
                                        n_results: int = 15, pca_dim: int = None,
                                        min_content_chars: int = 250) -> tuple:
        """
        Get recommendations with article content and measure query latency.

        Args:
            query: Urdu text query
            collection_key: Collection identifier ('full' or 'pca_64', etc.)
            n_results: Number of recommendations
            pca_dim: PCA dimension if using reduced embeddings
            min_content_chars: Minimum characters of content to extract

        Returns:
            Tuple of (results_with_content, latency_ms)
        """
        # Generate query embedding
        apply_pca = collection_key.startswith('pca_')
        query_embedding = self.generate_query_embedding(
            query, apply_pca=apply_pca, pca_dim=pca_dim
        )

        # Measure query latency
        start_time = time.time()
        results = self.collections[collection_key].query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results
        )
        latency = (time.time() - start_time) * 1000  # Convert to milliseconds

        # Extract article content for each result
        results_with_content = []
        if results['ids'] and len(results['ids'][0]) > 0:
            for i in range(len(results['ids'][0])):
                article_id = results['ids'][0][i]
                metadata = results['metadatas'][0][i]
                distance = results['distances'][0][i]
                similarity = 1 - distance
                
                # Get article content
                content = self.get_article_content(article_id, metadata, min_content_chars)
                
                results_with_content.append({
                    'rank': i + 1,
                    'article_id': article_id,
                    'headline': metadata.get('headline', 'N/A'),
                    'category': metadata.get('category', 'N/A'),
                    'similarity': similarity,
                    'distance': distance,
                    'content': content,
                    'content_length': len(content)
                })

        return results_with_content, latency

    def compare_all_embeddings_with_content(self, query: str, n_results: int = 15,
                                           min_content_chars: int = 250) -> dict:
        """
        Compare recommendations from Full (768D) and all PCA embeddings (64D, 128D, 256D) with content extraction.

        Args:
            query: Urdu text query
            n_results: Number of recommendations to return
            min_content_chars: Minimum characters of content per article

        Returns:
            Dictionary with all results, content, and latencies
        """
        print(f"\n{'='*70}")
        print(f"Query: {query[:100]}...")
        print(f"{'='*70}")

        comparison_data = {
            'query': query,
            'n_results': n_results,
            'min_content_chars': min_content_chars,
            'full': {},
            'pca_results': {}
        }

        # Get full embeddings results with content
        print(f"→ Full (768D)...")
        results_full, latency_full = self.get_recommendations_with_content(
            query, 'full', n_results, min_content_chars=min_content_chars
        )
        
        comparison_data['full'] = {
            'results': results_full,
            'latency': latency_full
        }
        print(f"  ✓ {len(results_full)} results | Latency: {latency_full:.2f}ms")

        # Get PCA results for each dimension
        for dim in self.pca_dimensions:
            print(f"→ PCA-{dim}D...")
            results_pca, latency_pca = self.get_recommendations_with_content(
                query, f'pca_{dim}', n_results, pca_dim=dim, min_content_chars=min_content_chars
            )

            comparison_data['pca_results'][dim] = {
                'results': results_pca,
                'latency': latency_pca
            }
            print(f"  ✓ {len(results_pca)} results | Latency: {latency_pca:.2f}ms")

        return comparison_data

    def create_docx_report(self, all_comparisons: list, output_path: Path):
        """
        Create comprehensive DOCX report with ONLY 64D recommendations and content.

        Args:
            all_comparisons: List of comparison dictionaries
            output_path: Output file path
        """
        print(f"\n{'='*70}")
        print("CREATING DOCX REPORT (64D ONLY)")
        print(f"{'='*70}")

        # Create document
        doc = Document()
        
        # Set default font
        style = doc.styles['Normal']
        font = style.font
        font.name = 'Arial'
        font.size = Pt(11)

        # Title
        title = doc.add_heading('Urdu News Recommendation System', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        subtitle = doc.add_heading('64D PCA Comparison Report', level=2)
        subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        info = doc.add_paragraph(f'Total Queries: {len(all_comparisons)}')
        info.alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        dimensions = doc.add_paragraph('Dimensions: Full (768D) vs PCA-64D')
        dimensions.alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        doc.add_paragraph()  # Empty line

        # Process each query
        for idx, comparison in enumerate(all_comparisons, 1):
            print(f"  → Processing Query {idx}/{len(all_comparisons)}")
            
            # Add page break except for first query
            if idx > 1:
                doc.add_page_break()
            
            # Query header
            doc.add_heading(f'Query {idx}', level=1)
            query_para = doc.add_paragraph()
            query_run = query_para.add_run(comparison['query'])
            query_run.italic = True
            query_run.font.size = Pt(12)
            
            doc.add_paragraph()  # Empty line

            # Performance summary table (Full 768D and PCA-64D only)
            doc.add_heading('Performance Summary', level=2)
            
            table = doc.add_table(rows=1, cols=3)
            table.style = 'Light Grid Accent 1'
            
            # Header row
            header_cells = table.rows[0].cells
            header_cells[0].text = 'Dimension'
            header_cells[1].text = 'Latency (ms)'
            header_cells[2].text = 'Results Count'
            
            # Make header bold
            for cell in header_cells:
                for paragraph in cell.paragraphs:
                    for run in paragraph.runs:
                        run.font.bold = True
            
            # Add Full 768D row
            full_data = comparison['full']
            row_cells = table.add_row().cells
            row_cells[0].text = 'Full (768D)'
            row_cells[1].text = f'{full_data["latency"]:.2f}'
            row_cells[2].text = str(len(full_data['results']))
            
            # Add ONLY PCA-64D row
            pca_64_data = comparison['pca_results'][64]
            row_cells = table.add_row().cells
            row_cells[0].text = 'PCA-64D'
            row_cells[1].text = f'{pca_64_data["latency"]:.2f}'
            row_cells[2].text = str(len(pca_64_data['results']))
            
            doc.add_paragraph()  # Empty line

            # Full 768D Recommendations
            doc.add_heading('Full (768D) Recommendations', level=2)
            
            full_results = comparison['full']['results']
            
            for result in full_results:
                # Rank and headline
                rank_para = doc.add_paragraph()
                rank_run = rank_para.add_run(f"Rank {result['rank']}: {result['headline']}")
                rank_run.font.bold = True
                rank_run.font.size = Pt(11)
                
                # Metadata
                meta_para = doc.add_paragraph()
                meta_text = (f"Similarity: {result['similarity']:.4f} | "
                           f"Category: {result['category']} | "
                           f"Content Length: {result['content_length']} chars")
                meta_run = meta_para.add_run(meta_text)
                meta_run.italic = True
                meta_run.font.size = Pt(10)
                meta_run.font.color.rgb = RGBColor(100, 100, 100)
                
                # Article content
                content_para = doc.add_paragraph(result['content'])
                content_para.style = 'Normal'
                
                doc.add_paragraph()  # Empty line between articles

            # PCA-64D Recommendations ONLY
            doc.add_heading('PCA-64D Recommendations', level=2)
            
            results_64 = comparison['pca_results'][64]['results']
            
            for result in results_64:
                # Rank and headline
                rank_para = doc.add_paragraph()
                rank_run = rank_para.add_run(f"Rank {result['rank']}: {result['headline']}")
                rank_run.font.bold = True
                rank_run.font.size = Pt(11)
                
                # Metadata
                meta_para = doc.add_paragraph()
                meta_text = (f"Similarity: {result['similarity']:.4f} | "
                           f"Category: {result['category']} | "
                           f"Content Length: {result['content_length']} chars")
                meta_run = meta_para.add_run(meta_text)
                meta_run.italic = True
                meta_run.font.size = Pt(10)
                meta_run.font.color.rgb = RGBColor(100, 100, 100)
                
                # Article content
                content_para = doc.add_paragraph(result['content'])
                content_para.style = 'Normal'
                
                doc.add_paragraph()  # Empty line between articles

        # Save document
        doc.save(output_path)
        print(f"  ✓ DOCX report saved: {output_path}")

    def batch_compare_queries_with_content(self, queries: list, n_results: int = 15,
                                          min_content_chars: int = 250,
                                          output_dir: str = "./Headline_PCA_Analysis"):
        """
        Compare multiple queries with content extraction and create DOCX report (64D only).

        Args:
            queries: List of Urdu queries
            n_results: Number of recommendations per query
            min_content_chars: Minimum characters of content per article
            output_dir: Directory to save results
        """
        output_path = Path(output_dir)
        output_path.mkdir(exist_ok=True)

        print(f"\n{'='*70}")
        print(f"BATCH COMPARISON WITH CONTENT: {len(queries)} QUERIES")
        print(f"{'='*70}")

        all_comparisons = []

        for idx, query in enumerate(queries, 1):
            print(f"\n{'='*70}")
            print(f"QUERY {idx}/{len(queries)}")
            print(f"{'='*70}")

            comparison = self.compare_all_embeddings_with_content(
                query, n_results, min_content_chars
            )
            all_comparisons.append(comparison)

        # Create comprehensive DOCX report (64D only)
        docx_path = output_path / 'content_sizes_comp_64d.docx'
        self.create_docx_report(all_comparisons, docx_path)

        return all_comparisons


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("MULTI-DIMENSIONAL URDU NEWS RECOMMENDATION SYSTEM")
    print("WITH CONTENT EXTRACTION (MIN 250 CHARS)")
    print("Comparing: Full (768D) vs PCA-64D")
    print("Using CLS pooling for embeddings")
    print("="*70)

    # Initialize recommender
    print(f"\n{'='*70}")
    print("INITIALIZING MULTI-DIMENSIONAL RECOMMENDER")
    print(f"{'='*70}")

    recommender = MultiDimensionalUrduNewsRecommender(
        model_name="urduhack/roberta-urdu-small",
        base_path="./chroma_db_headline_collections",
        pca_dimensions=[64, 128, 256]
    )

    # Define test queries - PASTE YOUR QUERIES HERE
    test_queries = [
    "ریاضی کا مشکل پریشان کن سوال",
    "پاکستان اسٹاک ایکسچینج کا ملا جلا رجحان",
    "سام سنگ کا نیا فلیگ شپ فون گلیکسی نوٹ",
    "سلمان خان اور دوستی کا پریشان کن مسئلہ",
    "ئی فون متعارف کرانے کی باضابطہ تاریخ",
    "پاکستان بمقابلہ ویسٹ انڈیز کرکٹ میچ کا نتیجہ",
    "ایشین اسنوکر چیمپئن شپ میں پاکستانی کیوسٹ",
    "یو ایس اوپن ٹینس میں نوواک جوکووچ کا مقابلہ",
    "پی ٹی اے زونگ جی اشتہارات واپس لینے کا حکم",
    "ناقدین کی نظر میں دنیا کی بہترین فلمیں",
    "پاکستانی کرکٹ اسکواڈ کا نیوزی لینڈ روانگی",
    "گوگل پر سب سے زیادہ سرچ ہونے والے ڈیوائسز",
    "پاکستان اسٹاک ایکسچینج میں پوائنٹس کا اضافہ",
    "پاکستان میں چالیس ایم بی رفتار انٹرنیٹ سروس",
    "تیل کی قیمتوں میں کمی کا فائدہ اٹھانے کا فیصلہ",
    "بھارتی اسٹار بیٹسمین راہول ڈریوڈ کا ریٹائرمنٹ اعلان",
    "جنوبی کوریا پاکستان سے تانبا معدنی اشیاء درامد",
    "جون میں ایل پی جی قیمت میں روپے کمی",
    "فلائی ویٹ چیمپئن محمد وسیم اٹھویں فائیٹ کی تیاری",
    "ایشوریہ رائے پر نسل پرستی کے سنگین الزامات",
    "ہالی ووڈ کی ہارر فلم گیس اسٹیشن لڑکی کہانی",
    "سام سنگ کے طاقتور گلیکسی ٹیبلیٹس کی رونمائی",
    "تاجروں کے دس رکنی وفد کا اس ماہ پاکستان دورہ",
    "ہالی ووڈ کی سنسنی خیز ڈرانی فلم بی فور ویک",
    "امریکا چاند پر اپنا پہلا بیس بنانے کا منصوبہ",
    "ہندوستانی خواتین کی نظر میں رنبیر کپور مثالی شوہر",
    "پاکستانی اداکار فواد خان بولڈ سینز کے بارے میں رائے",
    "ملکی معیشت بہتری کے لیے بجٹ میں سخت اقدامات",
    "نوکیا فولڈ ایبل فون بنانے کا حتمی فیصلہ",
    "اے سی کے ساتھ جانے والی عام غلطیاں اور حل",
    "بولی وڈ فلم فیئر گلیمر ایوارڈز کی رنگا رنگ تقریب",
    "خیبرپختونخواہ اٹھ ماہ کے بجٹ کا اعلان",
    "ایران پر بین الاقوامی پابندیوں کے خاتمے کا فیصلہ",
    "اسپیس ایکس کیپسول کی کامیاب واپسی امریکا میں",
    "پاکستان میں زیر گردش قرضے ارب روپے سے تجاوز",
    "یو ایس اوپن میں سرینا ولیمز اور اوساکا فائنل",
    "پاکستانی فاسٹ بالر محمد عامر میں سر اینڈی رابرٹس جھلک",
    "محمد عامر پر ڈومیسٹک کرکٹ میچ میں جرمانہ عائد",
    "بھارتی جاسوس کلبھوشن یادیو کے کیس کا فیصلہ",
    "پاکستانی اسٹارز تنقید اور تنازعات کی زد میں",
    "اداکارہ منال خان کے یورپ میں سیر سپاٹے کی تصاویر",
    "پاکستان ایشین ٹیم اسنوکر چیمپیئن شپ جیتنے میں کامیاب",
    "سام سنگ فون کا وہ فیچر جو ئی فون میں نہیں",
    "پنجاب بجٹ میں ملازمین کی تنخواہ میں اضافہ امکان",
    "پشاور جدید ٹیکنالوجی سے لیس کرائم سین پروٹیکشن یونٹ",
    "سام سنگ کا پہلا فولڈ ایبل فون اگلے ہفتے",
    "فیس بک اے ئی نظام کے ذریعے جادو کرنے کا دعوی",
    "پاکستان سپر لیگ کھلاڑیوں کی لاہور میں آمد",
    "ٹینس لیجنڈ بورس بیکر کے دیوالیہ ہونے کی خبر",
    "پاکستان انگلینڈ ٹیسٹ میں دونوں ٹیموں کے فٹنس مسائل",
    "سعید اجمل انگلش کاؤنٹی ووسٹر شائر میں شامل",
    "تھری ڈی پرنٹر نے دیو قامت کشتی چھاپ دی",
    "میک بک پرو میں اچانک آگ بھڑکنے کا واقعہ",
    "سماجی تحفظ کے لیے بجٹ دگنا کرنے کا فیصلہ",
    "فیس بک ملازمین کا امریکی صدر کی پوسٹس پر احتجاج",
    "پی سی بی کرکٹ کمیٹی کے چیئرمین اقبال قاسم استعفی",
    "کترینا کیف نے سلمان خان کو فلم کے لیے ٹھکرایا",
    "سلمان خان کی فلمیں ہٹ پر ہٹ جاری ہیں",
    "ای سی سی نے احساس پروگرام کے لیے ارب روپے منظوری",
    "ٹی سی پی نے ہزار ٹن سستی چینی فراہم کی",
    "پاکستان جنوبی افریقہ کے سامنے بے بس ہوا",
    "رمضان میں فروٹ چاٹ اور پکوڑوں کی مہنگائی",
    "ویرات کوہلی اور انوشکا شرما کی منگنی کی تصدیق",
    "گلوکار عالمگیر نے موت کی افواہوں کو مسترد کیا",
    "جاپان مزدوروں کی کمی دور کرنے کے لیے روبوٹ",
    "سوائن فلو کے دو مشتبہ مریضوں کی ہلاکت کی خبر",
    "زمبابوے کے برائن ویٹوری دنیا کے مہنگے ترین بالر",
    "کراچی سے خیبر تک پیٹرول کا مصنوعی بحران جاری",
    "پاکستان نے کینیڈا کو ہرا کر ہاکی ٹورنامنٹ جیتا",
    "بنگلہ دیش سری لنکا کے خلاف جیت کے لیے رنز درکار",
    "کورونا وائرس سے یورپ میں پاکستانی ٹیکسٹائل برامدات متاثر",
    "گلوکار علی ظفر کی ویں سالگرہ کی تقریبات",
    "دپیکا پڈوکون سلمان خان کے ساتھ فلم میں کام کرنا چاہتی ہیں",
    "پاکستان اور انگلینڈ کے درمیان دوسرا ٹیسٹ دبئی میں",
    "شاہ رخ خان پھر کترینہ کیف کے ساتھ فلم کریں گے",
    "نیپرا نے فیول ایڈجسٹمنٹ کی مد میں روپے اضافہ کردیا",
    "مائیکروسافٹ نے انٹرنیٹ ایکسپلورر سپورٹ بند کرنے کا اعلان",
    "موبائل کو کمپیوٹر پر ترجیح دینے والے صارفین",
    "سدھو کے پلوامہ حملے بیان پر کپل شرما شو میں ہٹائے گئے",
    "واٹس ایپ اسپام پیغامات روکنے کے لیے نیا فیچر",
    "ہالی ووڈ کی ایکشن ایڈونچر فلم اسیسنز کریڈ کا ٹریلر",
    "پی ایس ایل میں لاہور قلندرز بمقابلہ پشاور زلمی میچ",
    "جنوبی افریقہ کا تجارتی وفد اپریل میں پاکستان پہنچے گا",
    "شاہد افریدی نیوزی لینڈ میں اپنا آخری میچ کھیلیں گے",
    "ایشوریہ رائے بچن کی انتالیسویں سالگرہ کی تقریبات",
    "ایپل منفرد ئی فون بنانے کے منصوبوں پر کام",
    "بٹ کوائن ڈیجیٹل گولڈ کی قیمت میں ریکارڈ اضافہ",
    "سیف علی خان سوشل میڈیا پر موجود نہیں ہیں",
    "ویوو کمپنی کا گلیکسی نوٹ کو بھلا دینے والا فون",
    "فرنچ اوپن میں جووکووچ اور سمانتھا اسٹوزر دوسرے رانڈ میں",
    "حکومت نے صنعت کو کھولنے کا بروقت فیصلہ کیا",
    "انسٹاگرام فیڈ سروس میں مسائل کی وجہ سے شکایات",
    "فرانس میں ویں انٹرسلٹیک فیسٹیول کا بڑے پیمانے پر اغاز",
    "ملک بھر میں سونے کی قیمت میں دو سو روپے کمی",
    "ہالی ووڈ ایوارڈز تقریب کے دوران بارش کا پانی چھتریوں سے بہہ نکالا",
    "چین نے سیٹلائٹ مدار میں روانہ کر کے زمین کا سروے شروع کیا",
    "کراچی میں ورلڈ ونڈ انرجی کانفرنس اور نمائش کا انعقاد",
    "واٹس ایپ ایپلیکشن پرانے اپریٹنگ سسٹمز پر بند ہو رہی ہے",
    "پی ایس ایل فائنل کے یادگار لمحات اور فاتح ٹیم",
    "حفیظ شیخ اور رزاق داد سے اختلافات پر بورڈ چیئرمین استعفی",
]

    # Run batch comparison with content extraction
    if test_queries:
        print(f"\n{'='*70}")
        print("STARTING BATCH COMPARISON WITH CONTENT EXTRACTION")
        print(f"{'='*70}")

        all_comparisons = recommender.batch_compare_queries_with_content(
            queries=test_queries,
            n_results=15,
            min_content_chars=250,
            output_dir="./Headline_PCA_Analysis"
        )

        print(f"\n{'='*70}")
        print("✓ ALL COMPARISONS COMPLETED!")
        print(f"{'='*70}")
        print("Results saved in: ./Headline_PCA_Analysis/")
        print("  - DOCX Report: content_sizes_comp_64d.docx")
        print(f"{'='*70}")

    else:
        print("\nNo test queries provided. Please add queries to the test_queries list.")

    print(f"\n{'='*70}")
    print("✓ SYSTEM READY!")
    print(f"{'='*70}")
from matplotlib import rcParams
import time
import warnings
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
warnings.filterwarnings('ignore')

# Configure matplotlib
rcParams['figure.figsize'] = (16, 10)
rcParams['font.size'] = 10
sns.set_style("whitegrid")

class MultiDimensionalUrduNewsRecommender:
    """
    Recommendation system comparing Full (768D) with PCA-reduced embeddings (64D, 128D, 256D).
    Uses CLS pooling for embeddings.
    Includes query latency measurements and comprehensive overlap analysis.
    Extracts article content for each recommendation.
    """

    def __init__(self,
                 model_name: str = "urduhack/roberta-urdu-small",
                 base_path: str = "./chroma_db_headline_collections",
                 pca_dimensions: list = [64, 128, 256]):
        """
        Initialize recommender with connections to all embedding databases.

        Args:
            model_name: HuggingFace model identifier
            base_path: Base path for all ChromaDB collections
            pca_dimensions: List of PCA dimensions to compare
        """
        self.model_name = model_name
        self.base_path = Path(base_path)
        self.pca_dimensions = pca_dimensions
        self.pca_models = {}

        # Setup device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

        # Load PCA models
        for dim in pca_dimensions:
            pca_path = self.base_path / f"chroma_db_headline_pca_{dim}D" / "pca_model.pkl"
            if pca_path.exists():
                print(f"Loading PCA-{dim}D model from: {pca_path}")
                with open(pca_path, 'rb') as f:
                    self.pca_models[dim] = pickle.load(f)
                print(f"  ✓ PCA-{dim}D model loaded")
            else:
                print(f"  ⚠ Warning: PCA-{dim}D model not found")

        # Initialize ChromaDB clients and collections
        self.clients = {}
        self.collections = {}

        # Full embeddings collection
        full_path = self.base_path / "chroma_db_headline_full_768D"
        if full_path.exists():
            print(f"\nConnecting to Full Headline Embeddings (768D) at: {full_path}")
            self.clients['full'] = chromadb.PersistentClient(path=str(full_path))
            try:
                self.collections['full'] = self.clients['full'].get_collection(name="urdu_news_headline_full_768D_cls")
                print(f"  ✓ Connected: {self.collections['full'].count()} articles")
            except Exception as e:
                print(f"  ✗ Error: {e}")

        # PCA collections
        for dim in pca_dimensions:
            pca_path = self.base_path / f"chroma_db_headline_pca_{dim}D"
            if pca_path.exists():
                print(f"Connecting to PCA-{dim}D Headline at: {pca_path}")
                self.clients[f'pca_{dim}'] = chromadb.PersistentClient(path=str(pca_path))
                try:
                    self.collections[f'pca_{dim}'] = self.clients[f'pca_{dim}'].get_collection(
                        name=f"urdu_news_headline_pca_{dim}D_cls"
                    )
                    print(f"  ✓ Connected: {self.collections[f'pca_{dim}'].count()} articles")
                except Exception as e:
                    print(f"  ✗ Error: {e}")

    def cls_pooling(self, model_output, attention_mask=None):
        """
        Apply CLS pooling to get sentence embeddings.
        Returns the embedding of the [CLS] token.
        """
        return model_output.last_hidden_state[:, 0, :]

    def generate_query_embedding(self, query_text: str, max_length: int = 512,
                                 chunk_overlap: int = 50, apply_pca: bool = False,
                                 pca_dim: int = None) -> np.ndarray:
        """
        Generate embedding for query text using CLS pooling.
        Optionally applies PCA dimensionality reduction.

        Args:
            query_text: Urdu query text
            max_length: Maximum tokens per chunk
            chunk_overlap: Overlap between chunks
            apply_pca: If True, apply PCA reduction
            pca_dim: Which PCA dimension to apply (64, 128, or 256)

        Returns:
            Query embedding vector
        """
        tokens = self.tokenizer.encode(query_text, add_special_tokens=True)

        # Process short queries
        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                query_text, padding=True, truncation=True,
                max_length=max_length, return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.cls_pooling(model_output)
            embedding = embeddings.cpu().detach().numpy()[0]

            # Apply PCA if requested
            if apply_pca and pca_dim in self.pca_models:
                embedding = self.pca_models[pca_dim].transform(embedding.reshape(1, -1))[0]

            return embedding

        # For long queries: chunking
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text, padding=True, truncation=True,
                max_length=max_length, return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.cls_pooling(model_output)
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        final_embedding = np.mean(chunk_embeddings, axis=0)

        # Apply PCA if requested
        if apply_pca and pca_dim in self.pca_models:
            final_embedding = self.pca_models[pca_dim].transform(final_embedding.reshape(1, -1))[0]

        return final_embedding

    def get_article_content(self, article_id: str, metadata: dict, min_chars: int = 250) -> str:
        """
        Extract article content with minimum character requirement.

        Args:
            article_id: Article identifier
            metadata: Article metadata dictionary
            min_chars: Minimum characters to extract (default 250)

        Returns:
            Article content (at least min_chars if available)
        """
        # Try to get content from metadata
        content = metadata.get('content', '')
        
        # If content is empty, try to construct from available fields
        if not content:
            headline = metadata.get('headline', '')
            description = metadata.get('description', '')
            text = metadata.get('text', '')
            
            # Combine available text fields
            content = f"{headline}\n\n{description}\n\n{text}".strip()
        
        # If total content is less than min_chars, return what we have
        if len(content) < min_chars:
            return content if content else "محتویٰ دستیاب نہیں (Content not available)"
        
        # Return at least min_chars characters (or full content if it's close)
        return content

    def get_recommendations_with_content(self, query: str, collection_key: str,
                                        n_results: int = 15, pca_dim: int = None,
                                        min_content_chars: int = 250) -> tuple:
        """
        Get recommendations with article content and measure query latency.

        Args:
            query: Urdu text query
            collection_key: Collection identifier ('full' or 'pca_64', etc.)
            n_results: Number of recommendations
            pca_dim: PCA dimension if using reduced embeddings
            min_content_chars: Minimum characters of content to extract

        Returns:
            Tuple of (results_with_content, latency_ms)
        """
        # Generate query embedding
        apply_pca = collection_key.startswith('pca_')
        query_embedding = self.generate_query_embedding(
            query, apply_pca=apply_pca, pca_dim=pca_dim
        )

        # Measure query latency
        start_time = time.time()
        results = self.collections[collection_key].query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results
        )
        latency = (time.time() - start_time) * 1000  # Convert to milliseconds

        # Extract article content for each result
        results_with_content = []
        if results['ids'] and len(results['ids'][0]) > 0:
            for i in range(len(results['ids'][0])):
                article_id = results['ids'][0][i]
                metadata = results['metadatas'][0][i]
                distance = results['distances'][0][i]
                similarity = 1 - distance
                
                # Get article content
                content = self.get_article_content(article_id, metadata, min_content_chars)
                
                results_with_content.append({
                    'rank': i + 1,
                    'article_id': article_id,
                    'headline': metadata.get('headline', 'N/A'),
                    'category': metadata.get('category', 'N/A'),
                    'similarity': similarity,
                    'distance': distance,
                    'content': content,
                    'content_length': len(content)
                })

        return results_with_content, latency

    def compare_all_embeddings_with_content(self, query: str, n_results: int = 15,
                                           min_content_chars: int = 250) -> dict:
        """
        Compare recommendations from Full (768D) and all PCA embeddings (64D, 128D, 256D) with content extraction.

        Args:
            query: Urdu text query
            n_results: Number of recommendations to return
            min_content_chars: Minimum characters of content per article

        Returns:
            Dictionary with all results, content, and latencies
        """
        print(f"\n{'='*70}")
        print(f"Query: {query[:100]}...")
        print(f"{'='*70}")

        comparison_data = {
            'query': query,
            'n_results': n_results,
            'min_content_chars': min_content_chars,
            'full': {},
            'pca_results': {}
        }

        # Get full embeddings results with content
        print(f"→ Full (768D)...")
        results_full, latency_full = self.get_recommendations_with_content(
            query, 'full', n_results, min_content_chars=min_content_chars
        )
        
        comparison_data['full'] = {
            'results': results_full,
            'latency': latency_full
        }
        print(f"  ✓ {len(results_full)} results | Latency: {latency_full:.2f}ms")

        # Get PCA results for each dimension
        for dim in self.pca_dimensions:
            print(f"→ PCA-{dim}D...")
            results_pca, latency_pca = self.get_recommendations_with_content(
                query, f'pca_{dim}', n_results, pca_dim=dim, min_content_chars=min_content_chars
            )

            comparison_data['pca_results'][dim] = {
                'results': results_pca,
                'latency': latency_pca
            }
            print(f"  ✓ {len(results_pca)} results | Latency: {latency_pca:.2f}ms")

        return comparison_data

    def create_docx_report(self, all_comparisons: list, output_path: Path):
        """
        Create comprehensive DOCX report with all recommendations and content.

        Args:
            all_comparisons: List of comparison dictionaries
            output_path: Output file path
        """
        print(f"\n{'='*70}")
        print("CREATING DOCX REPORT")
        print(f"{'='*70}")

        # Create document
        doc = Document()
        
        # Set default font
        style = doc.styles['Normal']
        font = style.font
        font.name = 'Arial'
        font.size = Pt(11)

        # Title
        title = doc.add_heading('Urdu News Recommendation System', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        subtitle = doc.add_heading('Content Size Comparison Report', level=2)
        subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        info = doc.add_paragraph(f'Total Queries: {len(all_comparisons)}')
        info.alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        dimensions = doc.add_paragraph('Dimensions: Full (768D), PCA-64D, PCA-128D, PCA-256D')
        dimensions.alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        doc.add_paragraph()  # Empty line

        # Process each query
        for idx, comparison in enumerate(all_comparisons, 1):
            print(f"  → Processing Query {idx}/{len(all_comparisons)}")
            
            # Add page break except for first query
            if idx > 1:
                doc.add_page_break()
            
            # Query header
            doc.add_heading(f'Query {idx}', level=1)
            query_para = doc.add_paragraph()
            query_run = query_para.add_run(comparison['query'])
            query_run.italic = True
            query_run.font.size = Pt(12)
            
            doc.add_paragraph()  # Empty line

            # Performance summary table
            doc.add_heading('Performance Summary', level=2)
            
            table = doc.add_table(rows=1, cols=3)
            table.style = 'Light Grid Accent 1'
            
            # Header row
            header_cells = table.rows[0].cells
            header_cells[0].text = 'Dimension'
            header_cells[1].text = 'Latency (ms)'
            header_cells[2].text = 'Results Count'
            
            # Make header bold
            for cell in header_cells:
                for paragraph in cell.paragraphs:
                    for run in paragraph.runs:
                        run.font.bold = True
            
            # Add Full 768D row
            full_data = comparison['full']
            row_cells = table.add_row().cells
            row_cells[0].text = 'Full (768D)'
            row_cells[1].text = f'{full_data["latency"]:.2f}'
            row_cells[2].text = str(len(full_data['results']))
            
            # Add PCA rows
            for dim in self.pca_dimensions:
                pca_data = comparison['pca_results'][dim]
                row_cells = table.add_row().cells
                row_cells[0].text = f'PCA-{dim}D'
                row_cells[1].text = f'{pca_data["latency"]:.2f}'
                row_cells[2].text = str(len(pca_data['results']))
            
            doc.add_paragraph()  # Empty line

            # Full 768D Recommendations
            doc.add_heading('Full (768D) Recommendations', level=2)
            
            full_results = comparison['full']['results']
            
            for result in full_results:
                # Rank and headline
                rank_para = doc.add_paragraph()
                rank_run = rank_para.add_run(f"Rank {result['rank']}: {result['headline']}")
                rank_run.font.bold = True
                rank_run.font.size = Pt(11)
                
                # Metadata
                meta_para = doc.add_paragraph()
                meta_text = (f"Similarity: {result['similarity']:.4f} | "
                           f"Category: {result['category']} | "
                           f"Content Length: {result['content_length']} chars")
                meta_run = meta_para.add_run(meta_text)
                meta_run.italic = True
                meta_run.font.size = Pt(10)
                meta_run.font.color.rgb = RGBColor(100, 100, 100)
                
                # Article content
                content_para = doc.add_paragraph(result['content'])
                content_para.style = 'Normal'
                
                doc.add_paragraph()  # Empty line between articles

            # PCA Recommendations for each dimension
            for dim in self.pca_dimensions:
                doc.add_heading(f'PCA-{dim}D Recommendations', level=2)
                
                results = comparison['pca_results'][dim]['results']
                
                for result in results:
                    # Rank and headline
                    rank_para = doc.add_paragraph()
                    rank_run = rank_para.add_run(f"Rank {result['rank']}: {result['headline']}")
                    rank_run.font.bold = True
                    rank_run.font.size = Pt(11)
                    
                    # Metadata
                    meta_para = doc.add_paragraph()
                    meta_text = (f"Similarity: {result['similarity']:.4f} | "
                               f"Category: {result['category']} | "
                               f"Content Length: {result['content_length']} chars")
                    meta_run = meta_para.add_run(meta_text)
                    meta_run.italic = True
                    meta_run.font.size = Pt(10)
                    meta_run.font.color.rgb = RGBColor(100, 100, 100)
                    
                    # Article content
                    content_para = doc.add_paragraph(result['content'])
                    content_para.style = 'Normal'
                    
                    doc.add_paragraph()  # Empty line between articles

        # Save document
        doc.save(output_path)
        print(f"  ✓ DOCX report saved: {output_path}")

    def batch_compare_queries_with_content(self, queries: list, n_results: int = 15,
                                          min_content_chars: int = 250,
                                          output_dir: str = "./recommendation_results_with_content"):
        """
        Compare multiple queries with content extraction and create DOCX report.

        Args:
            queries: List of Urdu queries
            n_results: Number of recommendations per query
            min_content_chars: Minimum characters of content per article
            output_dir: Directory to save results
        """
        output_path = Path(output_dir)
        output_path.mkdir(exist_ok=True)

        print(f"\n{'='*70}")
        print(f"BATCH COMPARISON WITH CONTENT: {len(queries)} QUERIES")
        print(f"{'='*70}")

        all_comparisons = []

        for idx, query in enumerate(queries, 1):
            print(f"\n{'='*70}")
            print(f"QUERY {idx}/{len(queries)}")
            print(f"{'='*70}")

            comparison = self.compare_all_embeddings_with_content(
                query, n_results, min_content_chars
            )
            all_comparisons.append(comparison)

        # Create comprehensive DOCX report
        docx_path = output_path / 'content_sizes_comp.docx'
        self.create_docx_report(all_comparisons, docx_path)

        return all_comparisons


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("MULTI-DIMENSIONAL URDU NEWS RECOMMENDATION SYSTEM")
    print("WITH CONTENT EXTRACTION (MIN 250 CHARS)")
    print("Comparing: Full (768D), PCA (64D, 128D, 256D)")
    print("Using CLS pooling for embeddings")
    print("="*70)

    # Initialize recommender
    print(f"\n{'='*70}")
    print("INITIALIZING MULTI-DIMENSIONAL RECOMMENDER")
    print(f"{'='*70}")

    recommender = MultiDimensionalUrduNewsRecommender(
        model_name="urduhack/roberta-urdu-small",
        base_path="./chroma_db_headline_collections",
        pca_dimensions=[64, 128, 256]
    )

    # Define test queries - PASTE YOUR QUERIES HERE
    test_queries = [
    "ریاضی کے اس سوال کا جواب دے سکتے ہیں",
    "پاکستان اسٹاک ایکسچینج میں ملا جلا رجحان ہنڈرڈ انڈیکس پوائنٹس کمی پر بند",
    "سام سنگ کے نئے فلیگ شپ فون کی تاریخ رونمائی سامنے گئی",
    "سلمان خان کو دوستوں کی دوستی لے ڈوبی",
    "ئی فون کے متعارف کرانے کی تاریخ سامنے گئی",
    "پاکستان کے خلاف ویسٹ انڈیز قسمت بدلنے کا خواہاں",
    "ایشین اسنوکر چمپئن شپ میں پاکستانی کیوسٹ کا سفر ختم",
    "یو ایس اوپن ٹینس نوواک جوکووچ تیسرے رانڈ میں پہنچ گئے",
    "پی ٹی اے کی زونگ کو جی کے اشتہارات واپس لینے کی ہدایت",
    "ناقدین کی نظر میں ہر دور کی 15 بہترین فلمیں",
    "پاکستانی اسکواڈ کل کرائسٹ چرچ سے کوئنز ٹان روانہ ہو گا",
    "2015 میں سب سے زیادہ سرچ ہونے والی ڈیوائسز",
    "پاکستان اسٹاک ایکسچینج میں 90 پوائنٹس کا اضافہ",
    "اب پاکستان میں 40 سے 50 ایم بی رفتار انٹرنیٹ کی فراہمی ممکن ہوگئی",
    "حکومت کا تیل کی قیمتوں میں کمی کا فائدہ اٹھانے کا فیصلہ",
    "راہول ڈریوڈ نے انٹرنیشنل کرکٹ سے ریٹائرمنٹ کا اعلان کردیا",
    "جنوبی کوریا پاکستان سے تانبا معدنی اشیاء درامد کرے گا",
    "ایل پی جی کی قیمت میں جون سے 12 روپے فی کلو کمی کا اعلان",
    "فلائی ویٹ کے چیمپئن محمد وسیم نے اٹھویں فائیٹ کی تیاری کر لی",
    "ایشوریا رائے مشکل میںنسل پرستی کا الزام لگ گیا",
    "فلم کی کہانی گیس اسٹیشن پر کام کرنےوالی لڑکی کے گرد گھومتی ہے",
    "سام سنگ کے طاقتور ٹیبلیٹس متعارف",
    "تاجروں کا 10رکنی وفد اس ماہ پاکستان ائے گا",
    "ہالی ووڈ کی سنسنی خیز ڈرانی فلم بی فور ئی ویککاٹریلر جاری",
    "امریکا چاند پر بیس بنائے گا",
    "رنبیر کپور مثالی شوہر قرار",
    "بولڈ سینز کرنا نہ کرنا میری ذاتی مرضی ہے",
    "ائندہ بجٹ میں معیشت کی بہتری کے لیے اقدامات کیے جائیں گے اسد عمر",
    "نوکیا کا فولڈ ایبل فون بنانے کا فیصلہ",
    "اے سی کے ساتھ کی جانے والی 10 غلطیاں",
    "فلم فیئر گلیمر ایوارڈز کی رنگا رنگ تقریب",
    "خیبرپختونخواہ کا بجٹ پیش کردیا گیا",
    "ایران پر بین الاقوامی پابندیوں کا خاتمہ مشرق وسطی میں حصص کی منڈیوں میں کریش",
    "اسپیس ایکس کیپسول کی واپسی امریکا سے انسانی خلائی پروازوں کی راہ ہموار",
    "گردشی قرضے دوبارہ 400 ارب روپے سے تجاوز کرگئے",
    "یوایس اوپن امریکا کی سرینا ولیمز اور جاپان کی اوساکا فائنل میں پہنچ گئیں",
    "محمد عامر میں سر اینڈی رابرٹس کی جھلک نظر تی ہے کوچ سمرسیٹ کانٹی",
    "محمد عامر پر جرمانہ عائد",
    "کلبھوشن یادیو کا فیصلہ ارمی چیف میرٹ پرکرینگے ڈی جی ائی ایس پی ار",
    "رواں برس تنقید اور تنازعات کی زد میں رہنے والے پاکستانی اسٹارز",
    "اداکارہ منال خان کے یورپ میں سیر سپاٹے",
    "بھارت کو شکست پاکستان نے ایشین ٹیم اسنوکر چیمپیئن شپ جیت لی",
    "سام سنگ کے نئے فلیگ شپ فون کا وہ فیچر جو ئی فون میں نہیں",
    "پنجاب بجٹ اعداد شمار ملازمین کی تنخواہ میں 10 فیصد اضافے کا امکان",
    "پشاور جدید ٹیکنالوجی سے لیس کرائم سین پروٹیکشن یونٹ قائم",
    "سام سنگ کا پہلا فولڈ ایبل فون ایک ہفتے کی دوری پر",
    "اب فیس بک کے ساتھ جادو کرے گی",
    "کوئٹہ کے مقامی کھلاڑی لاہور پہنچ گئے غیرملکی کرکٹرز کل پہنچیں گے",
    "لندنٹینس لیجنڈ بورس بیکر دیوالیہ ہو گئے",
    "شارجہ ٹیسٹ پاکستان اور انگلش ٹیمیں فٹنس مسائل سے دوچار",
    "سہیل تنویرووسٹرشائر میں شامل",
    "تھری ڈی پرنٹر نے دیو قامت کشتی چھاپ دی",
    "میک بک پرو میں اچانک اگ بھڑک اٹھی",
    "سماجی تحفظ کے لیے بجٹ دگنا کرنے کا فیصلہ",
    "فیس بک ملازمین کا امریکی صدر کی پوسٹس پر کارروائی کا مطالبہ",
    "پی سی بی کرکٹ کمیٹی کے چیئرمین اقبال قاسم کا استعفی منظور",
    "کترینا کیف نے سلمان خان کو پھر ٹھکرادیا",
    "سلمان کی فلمیں ہٹ پر ہٹ رنبیر کپور مسلسل ناکام",
    "ای سی سی نے احساس پروگرام کے تحت 75 ارب روپے جاری کرنے کی منظوری دے دی",
    "ٹی سی پی نے 50 ہزار ٹن سستی چینی یوٹیلٹی اسٹورزکوفراہم کردی",
    "پاکستان جنوبی افریقہ کے سامنے بے بس",
    "مہنگائی کے باعث فروٹ چاٹ اور پکوڑے جیب پر بھاری پڑ گئے",
    "ویرات کوہلی اور انوشکا شرما کی منگنی کی اطلاعات",
    "معروف گلوکار عالمگیر نے موت کی افواہیں مسترد کردیں",
    "جاپان میں مزدوروں کی کمی دور کرنے کیلیے مستری روبوٹ تیار",
    "سوائن فلو کے دو مشتبہ مریضوں کی ہلاکت",
    "برائن ویٹوری ون ڈے میں دنیا کے چوتھے مہنگے ترین بالر",
    "کراچی سے خیبر تک پیٹرول کا مصنوعی بحران جاری",
    "پاکستان نے کینیڈا کو اذلان شاہ ہاکی ٹورنامنٹ میں ہرا کر فاتحانہ غاز کر دیا",
    "بنگلہ دیش کو سری لنکا کیخلاف جیت کیلئے390 رنز مزید درکار",
    "کورونا وائرس سے یورپ میں شٹ ڈان پاکستان کی ٹیکسٹائل برامدات متاثر",
    "گلوکار علی ظفر 37 سال کے ہوگئے",
    "دپیکاپڈوکون سلمان خان کیساتھ فلم میں کام کرنے کی خواہشمند",
    "پاکستان اور انگلینڈ ٹیسٹ سیریز دوسرا ٹیسٹ کل سے دبئی میں شروع ہورہاہے",
    "شاہ رخ خان پھر بنیں گے کترینہ کے ہیرو",
    "نیپرا نے فیول ایڈجسٹمنٹ کی مد میں فی یونٹ 198 روپے اضافہ کردیا",
    "انٹرنیٹ ایکسپلورر کا دور ختم ہونے کے قریب",
    "موبائل کو کمپیوٹر پر ترجیح",
    "پلوامہ حملے سے متعلق بیان سدھو کو کپل شرما شو سے ہاتھ دھونے پڑے",
    "واٹس ایپ پر غیرمتعلقہ پیغامات روکنے کیلئے نئے فیچر کی زمائش",
    "ایڈونچرسے بھرپورفلم اسیسنز کریڈکاٹریلر جاری",
    "پی ایس ایل اج لاہور قلندرز اور پشاور زلمی ٹکرائیں گے",
    "جنوبی افریقہ کا تجارتی وفد 11 اپریل کو پاکستان پہنچے گا",
    "نیوزی لینڈ میں کیرئیرکااخری میچ یادگاربناناچاہتا ہوںافریدی",
    "ایشوریہ رائے بچن انتالیسویں سالگرہ منارہی ہیں",
    "ایپل منفرد ائی فون بنانے کا خواہشمند",
    "ڈیجیٹل گولڈ کی قیمت میں ریکارڈ اضافہ",
    "گلیکسی نوٹ 10 کو بھلا دینے والا منفرد اسمارٹ فون",
    "حکومت نے سال میں بروقت ضروری اور تعمیراتی صنعت کو کھولا حماد اظہر",
    "انسٹاگرام کی فیڈ سروس متاثر صارفین کی شکایت پر کمپنی کی وضاحت",
    "فرانس میں48ویں انٹرسلٹیک فیسٹیول کا اغاز",
    "ملک بھر میں سونے کی قیمت میں کمی",
    "اسکر کی تقریب کے دوران بارش کا پانی چھتریوں سے بہہ نکالا",
    "چین نے سیٹلائٹ مدار میں روانہ کردیا زمین کا سروے کرے گا",
    "ورلڈ ونڈ انرجی کانفرنس اور نمائش اج سے کراچی میں شروع ہوگی",
    "واٹس ایپ ایپلیکشن پرانے اپریٹنگ سسٹمز پر بند",
    "پی ایس ایل فائنل 2017 یادگار لمحات",
    "حفیظ شیخ اور رزاق داد سے اختلافات چیئرمین سرمایہ کاری بورڈ زبیرگیلانی مستعفی"
    ]

    # Run batch comparison with content extraction
    if test_queries:
        print(f"\n{'='*70}")
        print("STARTING BATCH COMPARISON WITH CONTENT EXTRACTION")
        print(f"{'='*70}")

        all_comparisons = recommender.batch_compare_queries_with_content(
            queries=test_queries,
            n_results=15,
            min_content_chars=250,
            output_dir="./recommendation_results_with_content"
        )

        print(f"\n{'='*70}")
        print("✓ ALL COMPARISONS COMPLETED!")
        print(f"{'='*70}")
        print("Results saved in: ./recommendation_results_with_content/")
        print("  - DOCX Report: content_sizes_comp.docx")
        print(f"{'='*70}")

    else:
        print("\nNo test queries provided. Please add queries to the test_queries list.")

    print(f"\n{'='*70}")
    print("✓ SYSTEM READY!")
    print(f"{'='*70}")

MULTI-DIMENSIONAL URDU NEWS RECOMMENDATION SYSTEM
WITH CONTENT EXTRACTION (MIN 250 CHARS)
Comparing: Full (768D) vs PCA-64D
Using CLS pooling for embeddings

INITIALIZING MULTI-DIMENSIONAL RECOMMENDER
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Loading PCA-64D model from: chroma_db_headline_collections/chroma_db_headline_pca_64D/pca_model.pkl
  ✓ PCA-64D model loaded
Loading PCA-128D model from: chroma_db_headline_collections/chroma_db_headline_pca_128D/pca_model.pkl
  ✓ PCA-128D model loaded
Loading PCA-256D model from: chroma_db_headline_collections/chroma_db_headline_pca_256D/pca_model.pkl
  ✓ PCA-256D model loaded

Connecting to Full Headline Embeddings (768D) at: chroma_db_headline_collections/chroma_db_headline_full_768D
  ✓ Connected: 111853 articles
Connecting to PCA-64D Headline at: chroma_db_headline_collections/chroma_db_headline_pca_64D
  ✓ Connected: 111853 articles
Connecting to PCA-128D Headline at: chroma_db_headline_collections/chroma_db_headline_pca_